In [1]:
# Spotlight - Controlled Exploration Strategy
# Notebook 05

import pandas as pd
import numpy as np

print("Exploration strategy notebook ready!")

Exploration strategy notebook ready!


In [2]:
# Load dataset and recreate the temporal split

small_matrix = pd.read_csv("../data/small_matrix.csv")
item_features = pd.read_csv("../data/item_daily_features.csv")

# Same engagement score used throughout the project
small_matrix["watch_score"] = (
    small_matrix["watch_ratio"].clip(upper=3.0)
)

# Remove rows without a valid date
small_matrix = small_matrix.dropna(subset=["date"]).copy()

# Convert YYYYMMDD into datetime
small_matrix["date"] = pd.to_datetime(
    small_matrix["date"].astype(int).astype(str),
    format="%Y%m%d"
)

# Same temporal split used in previous notebooks
train_end = pd.Timestamp("2020-08-29")

train_data = small_matrix[
    small_matrix["date"] <= train_end
].copy()

test_data = small_matrix[
    small_matrix["date"] > train_end
].copy()

print("Training interactions:", len(train_data))
print("Testing interactions:", len(test_data))
print("Training users:", train_data["user_id"].nunique())
print("Training videos:", train_data["video_id"].nunique())

Training interactions: 4387382
Testing interactions: 107196
Training users: 1411
Training videos: 3288


In [3]:
# Create a clean video-to-tag mapping

video_tags = (
    item_features[
        ["video_id", "video_tag_name"]
    ]
    .dropna(subset=["video_tag_name"])
    .drop_duplicates("video_id")
)

# Keep only videos that appear in the interaction dataset
interaction_videos = small_matrix["video_id"].unique()

video_tags = video_tags[
    video_tags["video_id"].isin(interaction_videos)
].copy()

print("Videos in interaction dataset:", len(interaction_videos))
print("Videos with tags:", video_tags["video_id"].nunique())
print("Unique tags:", video_tags["video_tag_name"].nunique())
print("Missing tags:", video_tags["video_tag_name"].isna().sum())

Videos in interaction dataset: 3327
Videos with tags: 3324
Unique tags: 346
Missing tags: 0


In [4]:
# Build user-level interest profiles from training data only

user_history_tags = (
    train_data
    .merge(
        video_tags,
        on="video_id",
        how="left"
    )
    .dropna(subset=["video_tag_name"])
)

user_interest = (
    user_history_tags
    .groupby(
        ["user_id", "video_tag_name"]
    )
    .agg(
        interactions=("video_id", "count"),
        avg_watch_score=("watch_score", "mean")
    )
    .reset_index()
)

# User's overall average engagement
user_means = (
    train_data
    .groupby("user_id")["watch_score"]
    .mean()
    .rename("user_mean_watch_score")
    .reset_index()
)

user_interest = user_interest.merge(
    user_means,
    on="user_id",
    how="left"
)

# Smooth engagement so that one lucky/high-engagement
# interaction does not dominate the interest profile
smoothing = 5

user_interest["smoothed_interest"] = (
    (
        user_interest["interactions"]
        * user_interest["avg_watch_score"]
        + smoothing
        * user_interest["user_mean_watch_score"]
    )
    /
    (
        user_interest["interactions"]
        + smoothing
    )
)

print("Interest profile rows:", len(user_interest))
print("Users:", user_interest["user_id"].nunique())
print("Original tags:", user_interest["video_tag_name"].nunique())

Interest profile rows: 476973
Users: 1411
Original tags: 345


In [5]:
# Classify user interests by historical exposure

user_interest["interest_level"] = np.select(
    [
        user_interest["interactions"] >= 8,
        user_interest["interactions"] >= 1
    ],
    [
        "Established",
        "Weak"
    ],
    default="Unseen"
)

print(
    user_interest["interest_level"]
    .value_counts()
)

interest_level
Weak           359552
Established    117421
Name: count, dtype: int64


In [6]:
# Count weak and established interests for each user

user_interest_counts = (
    user_interest
    .groupby(["user_id", "interest_level"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

# Make sure both columns exist
for column in ["Weak", "Established"]:
    if column not in user_interest_counts.columns:
        user_interest_counts[column] = 0

user_interest_counts["total_known_interests"] = (
    user_interest_counts["Weak"]
    + user_interest_counts["Established"]
)

user_interest_counts["weak_interest_ratio"] = (
    user_interest_counts["Weak"]
    / user_interest_counts["total_known_interests"]
)

display(
    user_interest_counts[
        [
            "user_id",
            "Weak",
            "Established",
            "total_known_interests",
            "weak_interest_ratio"
        ]
    ].describe()
)

interest_level,user_id,Weak,Established,total_known_interests,weak_interest_ratio
count,1411.000000,1411.000000,1411.000000,1411.000000,1411.000000
mean,3631.627923,254.820695,83.218285,338.038979,0.753817
std,2044.494719,3.322568,2.065036,3.422584,0.005744
min,14.000000,240.000000,74.000000,324.000000,0.738095
25%,1835.000000,253.000000,82.000000,336.000000,0.750000
50%,3687.000000,255.000000,83.000000,338.000000,0.753666
75%,5419.000000,257.000000,85.000000,341.000000,0.757396
max,7162.000000,264.000000,88.000000,345.000000,0.781065


In [7]:
# Create controlled exploration candidates
# from each user's weakly explored interests

weak_interests = user_interest[
    user_interest["interest_level"] == "Weak"
][
    [
        "user_id",
        "video_tag_name",
        "smoothed_interest"
    ]
].copy()

# Connect each user's weak interests to videos
exploration_candidates = weak_interests.merge(
    video_tags[
        [
            "video_id",
            "video_tag_name"
        ]
    ],
    on="video_tag_name",
    how="inner"
)

print("Exploration candidate pairs:", len(exploration_candidates))
print(
    "Users with exploration candidates:",
    exploration_candidates["user_id"].nunique()
)
print(
    "Unique exploration videos:",
    exploration_candidates["video_id"].nunique()
)

Exploration candidate pairs: 920407
Users with exploration candidates: 1411
Unique exploration videos: 893


In [8]:
# Remove videos that the user has already watched during training

user_watched_videos = (
    train_data
    .groupby("user_id")["video_id"]
    .apply(set)
    .to_dict()
)

exploration_candidates["already_watched"] = [
    video_id in user_watched_videos[user_id]
    for user_id, video_id in zip(
        exploration_candidates["user_id"],
        exploration_candidates["video_id"]
    )
]

exploration_candidates_unseen = (
    exploration_candidates[
        ~exploration_candidates["already_watched"]
    ]
    .copy()
)

print(
    "Original exploration candidates:",
    len(exploration_candidates)
)

print(
    "Unseen exploration candidates:",
    len(exploration_candidates_unseen)
)

print(
    "Users with unseen exploration candidates:",
    exploration_candidates_unseen["user_id"].nunique()
)

print(
    "Unique unseen exploration videos:",
    exploration_candidates_unseen["video_id"].nunique()
)

Original exploration candidates: 920407
Unseen exploration candidates: 56302
Users with unseen exploration candidates: 1411
Unique unseen exploration videos: 752


In [9]:
# Assign an exploration score based on the user's
# learned interest in the corresponding weakly explored tag

exploration_candidates_unseen[
    "exploration_score"
] = exploration_candidates_unseen[
    "smoothed_interest"
]

print(
    exploration_candidates_unseen[
        "exploration_score"
    ].describe()
)

count    56302.000000
mean         0.850347
std          0.191138
min          0.258485
25%          0.724150
50%          0.830675
75%          0.957527
max          1.896770
Name: exploration_score, dtype: float64


In [10]:
# Exploration rates to evaluate

exploration_rates = [0.00, 0.10, 0.20, 0.30]

top_k = 10

print("Exploration rates:", exploration_rates)
print("Recommendation list size:", top_k)

Exploration rates: [0.0, 0.1, 0.2, 0.3]
Recommendation list size: 10


In [11]:
# Build the normal recommendation candidate pool
# using learned interest relevance.
#
# Previously watched videos are excluded.

# Create all user-video pairs for tagged videos
users = train_data["user_id"].unique()
videos = video_tags["video_id"].unique()

user_video_pairs = pd.MultiIndex.from_product(
    [users, videos],
    names=["user_id", "video_id"]
).to_frame(index=False)

# Attach video tags
normal_candidates = user_video_pairs.merge(
    video_tags[
        ["video_id", "video_tag_name"]
    ],
    on="video_id",
    how="left"
)

# Attach user interest scores
normal_candidates = normal_candidates.merge(
    user_interest[
        [
            "user_id",
            "video_tag_name",
            "smoothed_interest"
        ]
    ],
    on=["user_id", "video_tag_name"],
    how="left"
)

# No learned interest evidence = 0
normal_candidates["interest_relevance"] = (
    normal_candidates["smoothed_interest"]
    .fillna(0)
)

# Remove videos already watched during training
normal_candidates["already_watched"] = [
    video_id in user_watched_videos[user_id]
    for user_id, video_id in zip(
        normal_candidates["user_id"],
        normal_candidates["video_id"]
    )
]

normal_candidates = normal_candidates[
    ~normal_candidates["already_watched"]
].copy()

print(
    "Normal recommendation candidates:",
    len(normal_candidates)
)

print(
    "Users:",
    normal_candidates["user_id"].nunique()
)

print(
    "Videos:",
    normal_candidates["video_id"].nunique()
)

print(
    "Missing relevance scores:",
    normal_candidates["interest_relevance"].isna().sum()
)

Normal recommendation candidates: 306863
Users: 1411
Videos: 3264
Missing relevance scores: 0


In [12]:
# Create the normal interest-aware Top-10 recommendations

normal_recommendations = (
    normal_candidates
    .sort_values(
        ["user_id", "interest_relevance"],
        ascending=[True, False]
    )
    .groupby("user_id")
    .head(top_k)
    .reset_index(drop=True)
)

print(
    "Users with recommendations:",
    normal_recommendations["user_id"].nunique()
)

print(
    "Total recommendation slots:",
    len(normal_recommendations)
)

print(
    "Recommendations per user:",
    normal_recommendations
    .groupby("user_id")
    .size()
    .value_counts()
    .sort_index()
)


Users with recommendations: 1411
Total recommendation slots: 14110
Recommendations per user: 10    1411
Name: count, dtype: int64


In [13]:
# Create recommendation lists with controlled exploration

exploration_recommendations = {}

for rate in exploration_rates:

    exploration_slots = int(top_k * rate)
    normal_slots = top_k - exploration_slots

    user_recommendation_lists = []

    for user_id in users:

        # Normal recommendations
        normal_user = (
            normal_recommendations[
                normal_recommendations["user_id"] == user_id
            ]
            .sort_values(
                "interest_relevance",
                ascending=False
            )
            .head(normal_slots)
            .copy()
        )

        # Exploration recommendations
        exploration_user = (
            exploration_candidates_unseen[
                exploration_candidates_unseen["user_id"] == user_id
            ]
            .sort_values(
                "exploration_score",
                ascending=False
            )
            .head(exploration_slots)
            .copy()
        )

        # Mark recommendation source
        normal_user["recommendation_type"] = "Normal"
        exploration_user["recommendation_type"] = "Exploration"

        # Combine
        user_recommendations = pd.concat(
            [
                normal_user[
                    [
                        "user_id",
                        "video_id",
                        "recommendation_type"
                    ]
                ],
                exploration_user[
                    [
                        "user_id",
                        "video_id",
                        "recommendation_type"
                    ]
                ]
            ],
            ignore_index=True
        )

        user_recommendation_lists.append(
            user_recommendations
        )

    rate_recommendations = pd.concat(
        user_recommendation_lists,
        ignore_index=True
    )

    exploration_recommendations[rate] = (
        rate_recommendations
    )

    print(
        f"Exploration rate {rate:.0%}:",
        len(rate_recommendations),
        "recommendations"
    )

Exploration rate 0%: 14110 recommendations
Exploration rate 10%: 14110 recommendations
Exploration rate 20%: 14110 recommendations
Exploration rate 30%: 14110 recommendations


In [14]:
# Verify the number of normal and exploration slots
# at each exploration rate

exploration_composition = []

for rate, recommendations in exploration_recommendations.items():

    counts = (
        recommendations["recommendation_type"]
        .value_counts()
    )

    exploration_count = counts.get(
        "Exploration",
        0
    )

    normal_count = counts.get(
        "Normal",
        0
    )

    exploration_composition.append({
        "exploration_rate": rate,
        "normal_slots": normal_count,
        "exploration_slots": exploration_count,
        "total_slots": len(recommendations)
    })

exploration_composition = pd.DataFrame(
    exploration_composition
)

display(exploration_composition)

,exploration_rate,normal_slots,exploration_slots,total_slots
0,0.0,14110,0,14110
1,0.1,12699,1411,14110
2,0.2,11288,2822,14110
3,0.3,9877,4233,14110


In [15]:
# Verify that exploration recommendations come from weak interests

exploration_check = (
    exploration_candidates_unseen
    .groupby("user_id")
    .agg(
        exploration_videos=("video_id", "nunique"),
        mean_exploration_score=("exploration_score", "mean"),
        max_exploration_score=("exploration_score", "max")
    )
    .reset_index()
)

display(
    exploration_check[
        [
            "exploration_videos",
            "mean_exploration_score",
            "max_exploration_score"
        ]
    ].describe()
)

,exploration_videos,mean_exploration_score,max_exploration_score
count,1411.000000,1411.000000,1411.000000
mean,39.902197,0.850764,1.150957
std,11.817000,0.137746,0.202822
min,11.000000,0.422799,0.537963
25%,31.000000,0.756431,1.003780
50%,40.000000,0.838154,1.129657
75%,48.000000,0.931906,1.279022
max,88.000000,1.370972,1.896770


In [16]:
# Evaluate controlled exploration against the temporal test set

test_by_user = (
    test_data
    .groupby("user_id")["video_id"]
    .apply(set)
    .to_dict()
)

exploration_results = []

for rate, recommendations in exploration_recommendations.items():

    user_hit_rates = []
    exploration_hits = []
    exploration_slots = []

    for user_id, user_recs in recommendations.groupby("user_id"):

        actual_videos = test_by_user.get(
            user_id,
            set()
        )

        recommended_videos = set(
            user_recs["video_id"]
        )

        hits = len(
            recommended_videos.intersection(
                actual_videos
            )
        )

        user_hit_rates.append(
            hits / top_k
        )

        # Evaluate exploration slots separately
        exploration_recs = user_recs[
            user_recs["recommendation_type"]
            == "Exploration"
        ]

        exploration_slots.extend(
            exploration_recs["video_id"].tolist()
        )

        exploration_hits.extend(
            [
                video_id in actual_videos
                for video_id
                in exploration_recs["video_id"]
            ]
        )

    average_hit_rate = np.mean(
        user_hit_rates
    )

    exploration_hit_rate = (
        np.mean(exploration_hits)
        if len(exploration_hits) > 0
        else np.nan
    )

    exploration_results.append({
        "exploration_rate": rate,
        "average_hit_rate_at_10": average_hit_rate,
        "exploration_hit_rate": exploration_hit_rate,
        "total_recommendations": len(recommendations),
        "exploration_slots": len(exploration_slots)
    })

exploration_results = pd.DataFrame(
    exploration_results
)

display(exploration_results)

,exploration_rate,average_hit_rate_at_10,exploration_hit_rate,total_recommendations,exploration_slots
0,0.0,0.438767,NaN,14110,0
1,0.1,0.385471,0.223955,14110,1411
2,0.2,0.344862,0.240255,14110,2822
3,0.3,0.311623,0.244980,14110,4233


In [17]:
# Measure exploratory discovery / acceptance

exploration_acceptance = []

for rate, recommendations in exploration_recommendations.items():

    exploration_recs = recommendations[
        recommendations["recommendation_type"]
        == "Exploration"
    ].copy()

    if len(exploration_recs) == 0:
        continue

    exploration_recs["accepted"] = [
        video_id in test_by_user.get(user_id, set())
        for user_id, video_id in zip(
            exploration_recs["user_id"],
            exploration_recs["video_id"]
        )
    ]

    accepted_count = exploration_recs[
        "accepted"
    ].sum()

    total_exploration = len(
        exploration_recs
    )

    acceptance_rate = (
        accepted_count
        / total_exploration
        * 100
    )

    exploration_acceptance.append({
        "exploration_rate": rate,
        "exploration_recommendations": total_exploration,
        "accepted_explorations": accepted_count,
        "exploration_acceptance_rate": acceptance_rate
    })

exploration_acceptance = pd.DataFrame(
    exploration_acceptance
)

display(exploration_acceptance)

,exploration_rate,exploration_recommendations,accepted_explorations,exploration_acceptance_rate
0,0.1,1411,316,22.395464
1,0.2,2822,678,24.025514
2,0.3,4233,1037,24.497992


In [18]:
# Analyze creator groups receiving exploration exposure

# Create video -> creator mapping
video_creator = (
    item_features[
        ["video_id", "author_id"]
    ]
    .dropna(subset=["video_id", "author_id"])
    .drop_duplicates("video_id")
)

# Recreate creator popularity
creator_stats = (
    item_features
    .groupby("author_id")["show_cnt"]
    .sum()
    .reset_index(name="total_shows")
)

# Restrict creator statistics to experiment creators
experiment_creators = video_creator[
    video_creator["video_id"].isin(
        small_matrix["video_id"].unique()
    )
]["author_id"].unique()

creator_stats_exp = creator_stats[
    creator_stats["author_id"].isin(experiment_creators)
].copy()

# Define creator groups using the same threshold as Notebook 04
creator_threshold = creator_stats_exp[
    "total_shows"
].quantile(0.25)

creator_stats_exp["creator_group"] = np.where(
    creator_stats_exp["total_shows"]
    <= creator_threshold,
    "Emerging",
    "Established"
)

# Attach creator group to exploration recommendations
exploration_creator_results = []

for rate, recommendations in exploration_recommendations.items():

    exploration_recs = recommendations[
        recommendations["recommendation_type"]
        == "Exploration"
    ].copy()

    if len(exploration_recs) == 0:
        continue

    exploration_recs = exploration_recs.merge(
        video_creator,
        on="video_id",
        how="left"
    )

    exploration_recs = exploration_recs.merge(
        creator_stats_exp[
            ["author_id", "creator_group"]
        ],
        on="author_id",
        how="left"
    )

    emerging_count = (
        exploration_recs["creator_group"]
        .eq("Emerging")
        .sum()
    )

    established_count = (
        exploration_recs["creator_group"]
        .eq("Established")
        .sum()
    )

    exploration_creator_results.append({
        "exploration_rate": rate,
        "emerging_exploration_slots": emerging_count,
        "established_exploration_slots": established_count,
        "emerging_exploration_exposure": (
            emerging_count
            / len(exploration_recs)
            * 100
        )
    })

exploration_creator_results = pd.DataFrame(
    exploration_creator_results
)

display(exploration_creator_results)

,exploration_rate,emerging_exploration_slots,established_exploration_slots,emerging_exploration_exposure
0,0.1,117,1294,8.291991
1,0.2,246,2576,8.717222
2,0.3,367,3866,8.669974


In [19]:
# Add creator information and fairness boost
# to unseen exploration candidates

exploration_fairness_candidates = (
    exploration_candidates_unseen
    .merge(
        video_creator[
            ["video_id", "author_id"]
        ],
        on="video_id",
        how="left"
    )
    .merge(
        creator_stats_exp[
            [
                "author_id",
                "total_shows",
                "creator_group"
            ]
        ],
        on="author_id",
        how="left"
    )
)

# Population-matching fairness strength from Notebook 04
exploration_fairness_strength = 0.08

exploration_fairness_candidates[
    "fairness_boost"
] = np.where(
    exploration_fairness_candidates[
        "creator_group"
    ] == "Emerging",
    exploration_fairness_strength,
    0
)

exploration_fairness_candidates[
    "combined_exploration_score"
] = (
    exploration_fairness_candidates[
        "exploration_score"
    ]
    +
    exploration_fairness_candidates[
        "fairness_boost"
    ]
)

print(
    "Exploration fairness candidates:",
    len(exploration_fairness_candidates)
)

print(
    "Missing creator groups:",
    exploration_fairness_candidates[
        "creator_group"
    ].isna().sum()
)

print(
    "Emerging candidates:",
    (
        exploration_fairness_candidates[
            "creator_group"
        ] == "Emerging"
    ).sum()
)

print(
    "Established candidates:",
    (
        exploration_fairness_candidates[
            "creator_group"
        ] == "Established"
    ).sum()
)

Exploration fairness candidates: 56302
Missing creator groups: 0
Emerging candidates: 5001
Established candidates: 51301


In [20]:
# Create fairness-aware exploration recommendations

fair_exploration_recommendations = []

for user_id in users:

    user_candidates = (
        exploration_fairness_candidates[
            exploration_fairness_candidates["user_id"]
            == user_id
        ]
        .sort_values(
            "combined_exploration_score",
            ascending=False
        )
    )

    # Maximum exploration slots needed = 3
    top_user_exploration = user_candidates.head(3).copy()

    top_user_exploration[
        "recommendation_type"
    ] = "Exploration"

    fair_exploration_recommendations.append(
        top_user_exploration[
            [
                "user_id",
                "video_id",
                "author_id",
                "creator_group",
                "exploration_score",
                "fairness_boost",
                "combined_exploration_score",
                "recommendation_type"
            ]
        ]
    )

fair_exploration_recommendations = pd.concat(
    fair_exploration_recommendations,
    ignore_index=True
)

print(
    "Fairness-aware exploration recommendations:",
    len(fair_exploration_recommendations)
)

print(
    "Users:",
    fair_exploration_recommendations[
        "user_id"
    ].nunique()
)

print(
    "Recommendations per user:"
)

print(
    fair_exploration_recommendations
    .groupby("user_id")
    .size()
    .value_counts()
    .sort_index()
)

Fairness-aware exploration recommendations: 4233
Users: 1411
Recommendations per user:
3    1411
Name: count, dtype: int64


In [21]:
# Measure creator-group exposure after applying
# the fairness boost to exploration candidates

fair_exploration_exposure = (
    fair_exploration_recommendations[
        "creator_group"
    ]
    .value_counts()
    .reset_index()
)

fair_exploration_exposure.columns = [
    "creator_group",
    "exploration_slots"
]

fair_exploration_exposure[
    "exposure_percentage"
] = (
    fair_exploration_exposure[
        "exploration_slots"
    ]
    / len(fair_exploration_recommendations)
    * 100
)

display(fair_exploration_exposure)

,creator_group,exploration_slots,exposure_percentage
0,Established,3430,81.030002
1,Emerging,803,18.969998


In [22]:
# Evaluate fairness-aware exploration using
# subsequent test-period interactions

fair_exploration_recs = (
    fair_exploration_recommendations.copy()
)

fair_exploration_recs["accepted"] = [
    video_id in test_by_user.get(
        user_id,
        set()
    )
    for user_id, video_id in zip(
        fair_exploration_recs["user_id"],
        fair_exploration_recs["video_id"]
    )
]

accepted_count = (
    fair_exploration_recs["accepted"]
    .sum()
)

total_exploration = len(
    fair_exploration_recs
)

overall_acceptance_rate = (
    accepted_count
    / total_exploration
    * 100
)

# Acceptance by creator group
group_acceptance = (
    fair_exploration_recs
    .groupby("creator_group")
    .agg(
        exploration_recommendations=(
            "video_id",
            "count"
        ),
        accepted_explorations=(
            "accepted",
            "sum"
        )
    )
    .reset_index()
)

group_acceptance[
    "acceptance_rate"
] = (
    group_acceptance["accepted_explorations"]
    / group_acceptance[
        "exploration_recommendations"
    ]
    * 100
)

print(
    "Total fairness-aware exploration:",
    total_exploration
)

print(
    "Accepted explorations:",
    accepted_count
)

print(
    "Overall exploration acceptance rate:",
    round(
        overall_acceptance_rate,
        4
    )
)

display(group_acceptance)

Total fairness-aware exploration: 4233
Accepted explorations: 943
Overall exploration acceptance rate: 22.2773


,creator_group,exploration_recommendations,accepted_explorations,acceptance_rate
0,Emerging,803,68,8.468244
1,Established,3430,875,25.510204


### Fairness-Aware Exploration Acceptance

The fairness-aware exploration strategy generated 4,233 exploratory recommendations, of which 943 were subsequently observed in the test-period interactions, giving an overall exploration acceptance rate of 22.28%.

The exposure distribution was shifted toward Emerging creators: 803 recommendations (18.97%) came from Emerging creators and 3,430 (81.03%) from Established creators.

However, acceptance differed substantially between the two creator groups. Emerging creator recommendations had an acceptance rate of 8.47%, while Established creator recommendations had an acceptance rate of 25.51%.

This indicates an important relevance–fairness trade-off. Increasing exposure to Emerging creators does not automatically result in equivalent user acceptance. Therefore, a fairness-aware recommendation system should incorporate both creator exposure and user-interest signals rather than optimizing exposure alone.

The acceptance measure used here is a behavioral proxy: a recommendation is considered accepted when the corresponding video appears in the user's subsequent test-period interactions. It should not be interpreted as an explicit like/dislike signal.

In [23]:
# Creator diversity in fairness-aware exploration

creator_diversity = (
    fair_exploration_recommendations
    .groupby("user_id")["author_id"]
    .nunique()
)

print("Mean unique creators per user:", round(creator_diversity.mean(), 4))
print("Median unique creators per user:", round(creator_diversity.median(), 4))
print("Minimum unique creators:", creator_diversity.min())
print("Maximum unique creators:", creator_diversity.max())

Mean unique creators per user: 2.9872
Median unique creators per user: 3.0
Minimum unique creators: 1
Maximum unique creators: 3


In [24]:
# Final comparison of exploration strategies

exploration_comparison = pd.DataFrame({
    "Exploration Rate": ["0%", "10%", "20%", "30%"],
    "Overall HitRate@10": [0.438767, 0.385471, 0.344862, 0.311623],
    "Exploration Acceptance": [np.nan, 22.395464, 24.025514, 24.497992],
    "Exploration Slots": [0, 1411, 2822, 4233]
})

display(exploration_comparison)

,Exploration Rate,Overall HitRate@10,Exploration Acceptance,Exploration Slots
0,0%,0.438767,NaN,0
1,10%,0.385471,22.395464,1411
2,20%,0.344862,24.025514,2822
3,30%,0.311623,24.497992,4233


# Notebook 05 Summary — Controlled Exploration Strategy

## What was established

This notebook developed a controlled exploration mechanism for introducing users to videos outside their strongest established interests.

Instead of recommending completely random content, exploration was restricted to videos associated with the user's weak interests (1–7 historical interactions). Previously watched videos were removed so that exploration represented genuinely new video exposure.

The exploration mechanism was evaluated at 0%, 10%, 20%, and 30% exploration rates.

## Key Results

| Metric | Result |
|---|---:|
| Weak-interest candidate pairs | 920,407 |
| Unseen exploration candidate pairs | 56,302 |
| Users with exploration candidates | 1,411 |
| Unique unseen exploration videos | 752 |
| 10% exploration HitRate@10 | 38.55% |
| 20% exploration HitRate@10 | 34.49% |
| 30% exploration HitRate@10 | 31.16% |
| 10% exploration acceptance | 22.40% |
| 20% exploration acceptance | 24.03% |
| 30% exploration acceptance | 24.50% |

The fairness-aware exploration strategy used the fairness strength identified in Notebook 04 as the population-matching setting (0.08). It increased Emerging creator exposure within exploration from approximately 8–9% to 18.97%.

However, acceptance differed between creator groups:

- Emerging creators: 8.47% acceptance
- Established creators: 25.51% acceptance
- Overall: 22.28% acceptance

The fairness-aware exploration generated 4,233 exploratory recommendations, with 943 subsequently observed in the test-period interactions.

Creator diversity was also high: users received an average of 2.99 unique creators across their three fairness-aware exploratory recommendations, with a median of 3.

## Research Implication

The results demonstrate a measurable trade-off between recommendation relevance, exploration, and creator exposure.

Increasing exploration allows the system to introduce users to content beyond their strongest interests, and the exploratory recommendations show non-zero behavioral acceptance. However, increasing the exploration rate reduces overall HitRate@10.

The fairness-aware exploration experiment further shows that increasing exposure to Emerging creators does not automatically produce equivalent user acceptance. Therefore, fairness should not be treated as an independent exposure objective. A practical recommendation system should jointly consider user interest, exploration, creator exposure, and behavioral acceptance.

The exploration strategy therefore provides the foundation for the final evaluation stage, where different recommendation configurations can be compared using relevance, fairness, creator coverage, diversity, and exploration-related metrics.

## Important Methodological Note

Exploration acceptance is defined as the proportion of exploratory recommendations whose videos appear in the user's subsequent test-period interactions. This is a behavioral proxy for acceptance and should not be interpreted as an explicit like/dislike measurement.

The current experiment is an offline simulation of controlled exploration. The KuaiRec dataset does not provide a true online feedback loop in which the recommendation system observes a user's response and immediately updates the next recommendation. Therefore, sequential online adaptation is not claimed by this experiment.

## Notebook Status

Notebook 05 is complete.

Established:
1. Controlled weak-interest exploration
2. Exploration/relevance trade-off
3. Exploration acceptance measurement
4. Creator exposure under exploration
5. Fairness-aware exploration
6. Creator-level diversity of exploratory recommendations

**Notebooks remaining: 2 — Notebook 06 (Evaluation) and Notebook 07 (Results Visualization).**